<div style="padding:2.2rem;border-radius:18px;background:linear-gradient(135deg,#172554,#0f766e);color:white;">
<p style="font-size:1.05rem;letter-spacing:.12em;text-transform:uppercase;opacity:.85;">D085 · Data Engineering</p>
<h1 style="font-size:3rem;margin:.3rem 0;">OLTP with SQLite</h1>
<p style="font-size:1.4rem;max-width:900px;">Build the first table of an e-commerce database and learn keys, constraints, CRUD, and safe failure handling.</p>
<hr style="border:0;border-top:1px solid rgba(255,255,255,.35);margin:2rem 0;">
<p>Python · SQL · SQLite · One table first</p>
</div>

# Learning objectives

By the end of this notebook, you can:

- Explain why an e-commerce application uses an **OLTP** database
- Create a table with a primary key, unique key, defaults, and validation constraints
- Use a generated sequential key
- Run `INSERT`, `SELECT`, `UPDATE`, and `DELETE` statements
- Read common constraint errors and recover safely
- Explain why customers, orders, and products should eventually be separate tables

> Scope: we create only the `products` table today. The next notebook will add related tables and foreign keys.

# The e-commerce database we are growing

A future normalized design might contain:

```text
customers ──< orders ──< order_items >── products
```

We will grow toward that design gradually. In this notebook, the database contains **one entity only**:

```text
products
├── product_id       generated primary key
├── sku              business identifier; unique
├── product_name     required
├── category         required
├── unit_price       non-negative
├── stock_quantity   non-negative
├── is_active        0 or 1
└── created_at       generated timestamp
```

# Why start with one table?

A tempting first design is one large spreadsheet-like table:

| order_id | customer_name | customer_email | product_name | product_price | quantity |
|---:|---|---|---|---:|---:|
| 101 | Asha | asha@example.com | Keyboard | 2499 | 1 |
| 102 | Asha | asha@example.com | Mouse | 899 | 2 |

This repeats customer and product facts. Repetition creates anomalies:

- **Update anomaly:** changing Asha's email requires several row updates.
- **Insert anomaly:** a product cannot be recorded until an order exists.
- **Delete anomaly:** deleting the last order may erase the only product record.

**Normalization** separates facts about different entities. We begin with `products`; later notebooks will introduce the relationships safely.

# OLTP in one minute

**Online Transaction Processing (OLTP)** systems handle many small, current business operations: add a product, change stock, place an order, or update a customer address.

Typical OLTP properties:

- Many short inserts and updates
- Fast lookup of individual records
- Constraints that protect correctness
- Transactions that succeed or fail as one unit
- A normalized model that reduces duplication

SQLite is a small embedded relational database. It is excellent for learning SQL and for local applications.

In [ ]:
import sqlite3

print("SQLite version:", sqlite3.sqlite_version)

# :memory: creates a fresh temporary database for this notebook session.
connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
cursor = connection.cursor()

# Keys and constraints

| Feature | Purpose | Example |
|---|---|---|
| `PRIMARY KEY` | Identifies each row; cannot be duplicated | `product_id` |
| `AUTOINCREMENT` | Generates an increasing integer key | 1, 2, 3, ... |
| `UNIQUE` | Prevents duplicate business values | `sku` |
| `NOT NULL` | Requires a value | `product_name` |
| `CHECK` | Allows only valid values | `unit_price >= 0` |
| `DEFAULT` | Supplies a value when omitted | `is_active = 1` |

`product_id` is a **surrogate key** generated by the database. `sku` is a **business key** meaningful to the company. Keeping both lets the internal identity remain stable even if a business code changes.

In [ ]:
cursor.execute("""
CREATE TABLE products (
    product_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    sku            TEXT NOT NULL UNIQUE,
    product_name   TEXT NOT NULL,
    category       TEXT NOT NULL,
    unit_price     REAL NOT NULL CHECK (unit_price >= 0),
    stock_quantity INTEGER NOT NULL DEFAULT 0 CHECK (stock_quantity >= 0),
    is_active      INTEGER NOT NULL DEFAULT 1 CHECK (is_active IN (0, 1)),
    created_at     TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);
""")
connection.commit()
print("Created table: products")

# Inspect the schema

Database metadata is useful when we inherit an unfamiliar database. SQLite exposes table information through `PRAGMA` commands.

In [ ]:
schema = cursor.execute("PRAGMA table_info(products)").fetchall()
for column in schema:
    print(dict(column))

# CREATE: insert rows

Use placeholders (`?`) instead of building SQL with string concatenation. Parameters correctly handle quoting and help prevent SQL injection.

In [ ]:
products_to_insert = [
    ("ELEC-KB-001", "Mechanical Keyboard", "Electronics", 2499.00, 25),
    ("ELEC-MS-001", "Wireless Mouse", "Electronics", 899.00, 40),
    ("HOME-MG-001", "Ceramic Coffee Mug", "Home", 349.00, 60),
    ("BOOK-SQL-001", "Practical SQL", "Books", 799.00, 15),
]

cursor.executemany("""
    INSERT INTO products
        (sku, product_name, category, unit_price, stock_quantity)
    VALUES (?, ?, ?, ?, ?)
""", products_to_insert)
connection.commit()

print("Rows inserted:", cursor.rowcount)

# READ: select rows

`SELECT` chooses columns and rows. `WHERE` filters; `ORDER BY` controls presentation.

In [ ]:
rows = cursor.execute("""
    SELECT product_id, sku, product_name, unit_price, stock_quantity
    FROM products
    ORDER BY product_id
""").fetchall()

for row in rows:
    print(dict(row))

In [ ]:
maximum_price = 1000
affordable_products = cursor.execute("""
    SELECT product_name, unit_price
    FROM products
    WHERE unit_price < ? AND is_active = 1
    ORDER BY unit_price
""", (maximum_price,)).fetchall()

for row in affordable_products:
    print(dict(row))

# UPDATE: change an existing row

Always test the `WHERE` condition with a `SELECT` first. Without `WHERE`, every row is updated.

In [ ]:
cursor.execute("""
    UPDATE products
    SET unit_price = ?, stock_quantity = stock_quantity + ?
    WHERE sku = ?
""", (849.00, 10, "ELEC-MS-001"))
connection.commit()

print("Rows updated:", cursor.rowcount)
updated = cursor.execute(
    "SELECT * FROM products WHERE sku = ?", ("ELEC-MS-001",)
).fetchone()
print(dict(updated))

# DELETE: remove a row

A delete is permanent after commit. In business systems, products are often deactivated instead, preserving their history.

In [ ]:
# Insert a temporary row so the example is safe and repeatable.
cursor.execute("""
    INSERT INTO products (sku, product_name, category, unit_price)
    VALUES (?, ?, ?, ?)
""", ("DEMO-DELETE-001", "Temporary Demo Product", "Demo", 1.00))

cursor.execute("DELETE FROM products WHERE sku = ?", ("DEMO-DELETE-001",))
connection.commit()
print("Rows deleted:", cursor.rowcount)

# Constraint failures are useful

Constraints reject invalid data close to where it is stored. The helper below uses a **savepoint**, attempts bad SQL, prints the database error, and rolls back only that attempt. These are expected learning examples.

In [ ]:
def expect_constraint_error(label, sql, parameters=()):
    cursor.execute("SAVEPOINT constraint_demo")
    try:
        cursor.execute(sql, parameters)
    except sqlite3.IntegrityError as error:
        print(f"{label}: {error}")
    else:
        print(f"{label}: no error (check the example)")
    finally:
        cursor.execute("ROLLBACK TO constraint_demo")
        cursor.execute("RELEASE constraint_demo")

In [ ]:
# UNIQUE failure: this SKU already exists.
expect_constraint_error(
    "Duplicate unique key",
    """INSERT INTO products
       (sku, product_name, category, unit_price)
       VALUES (?, ?, ?, ?)""",
    ("ELEC-KB-001", "Another Keyboard", "Electronics", 1999.00),
)

# NOT NULL failure: product_name is required.
expect_constraint_error(
    "Missing required value",
    """INSERT INTO products
       (sku, product_name, category, unit_price)
       VALUES (?, ?, ?, ?)""",
    ("BAD-NULL-001", None, "Demo", 100.00),
)

# CHECK failure: stock cannot be negative.
expect_constraint_error(
    "Invalid stock",
    "UPDATE products SET stock_quantity = ? WHERE sku = ?",
    (-5, "ELEC-MS-001"),
)

# PRIMARY KEY failure: product_id 1 already exists.
expect_constraint_error(
    "Duplicate primary key",
    """INSERT INTO products
       (product_id, sku, product_name, category, unit_price)
       VALUES (?, ?, ?, ?, ?)""",
    (1, "BAD-PK-001", "Duplicate ID Product", "Demo", 10.00),
)

# Sequential keys and `sqlite_sequence`

`INTEGER PRIMARY KEY AUTOINCREMENT` asks SQLite to choose an integer larger than any previously generated key for this table. SQLite records the last generated value in an internal table named `sqlite_sequence`.

Important:

- A generated key is unique identity, not a row count.
- Deleted key values are not automatically reused.
- Gaps are normal; never promise customers gap-free IDs.
- Applications should usually omit the generated key during insert.

In [ ]:
sequence = cursor.execute("""
    SELECT name, seq
    FROM sqlite_sequence
    WHERE name = 'products'
""").fetchone()
print(dict(sequence))

ids = cursor.execute(
    "SELECT product_id, sku FROM products ORDER BY product_id"
).fetchall()
print([dict(row) for row in ids])

# Safer updates with transactions

A transaction groups related changes. Either all changes commit, or a rollback restores the previous state. Here a sale reduces stock only when enough inventory exists.

In [ ]:
def sell_product(sku, quantity):
    if quantity <= 0:
        raise ValueError("quantity must be positive")

    try:
        cursor.execute("BEGIN")
        cursor.execute("""
            UPDATE products
            SET stock_quantity = stock_quantity - ?
            WHERE sku = ?
              AND is_active = 1
              AND stock_quantity >= ?
        """, (quantity, sku, quantity))

        if cursor.rowcount != 1:
            raise ValueError("product missing, inactive, or stock is insufficient")

        connection.commit()
        print(f"Sale completed: {quantity} unit(s) of {sku}")
    except Exception:
        connection.rollback()
        raise

sell_product("ELEC-KB-001", 2)
print(dict(cursor.execute(
    "SELECT sku, stock_quantity FROM products WHERE sku = ?",
    ("ELEC-KB-001",),
).fetchone()))

# Practice

Complete these exercises by adding cells below:

1. Insert a new product and let SQLite generate its ID and timestamp.
2. Select all active products with stock below 30.
3. Increase every product in the `Books` category by 10%, using a `WHERE` clause.
4. Deactivate a product by setting `is_active = 0` instead of deleting it.
5. Try inserting a negative price. Catch and explain the error.
6. Explain why `sku` alone might be a poor primary key even though it is unique.

**Challenge:** write a query that reports each category, number of products, and total stock using `GROUP BY`.

# Recap and next step

Today we used one normalized entity and learned:

- A primary key gives every row a stable identity.
- A unique key protects a business identifier.
- `NOT NULL`, `CHECK`, and `DEFAULT` protect data quality.
- Generated keys may contain gaps and should not carry business meaning.
- CRUD means Create, Read, Update, and Delete.
- Transactions and rollbacks protect multi-step operations.

**Next notebook:** add `customers`, `orders`, and `order_items`; connect them with foreign keys; then observe referential-integrity successes and failures.

In [ ]:
# Release the in-memory database when finished.
connection.close()
print("Database connection closed.")